# 🪪 Face ID System — Colab Launcher

**Steps:**
1. Run Cell 1 → installs all dependencies (GPU-accelerated)
2. Run Cell 2 → uploads your project files
3. Run Cell 3 → mounts Google Drive (saves your face database persistently)
4. Run Cell 4 → starts the Streamlit app and gives you a public URL

> ✅ Runtime → **T4 GPU** recommended for fast FaceNet inference


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
print('📦 Installing dependencies...')
!pip install -q streamlit pyngrok opencv-python-headless facenet-pytorch Pillow
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
print('✅ Dependencies installed!')

In [ ]:
# ── Cell 2: Upload project files ──────────────────────────────────────────────
# Upload: app.py, model.py, recognition.py, database.py
import os
from google.colab import files

os.makedirs('/content/faceid', exist_ok=True)
os.chdir('/content/faceid')

print('📂 Upload your 4 project files: app.py, model.py, recognition.py, database.py')
uploaded = files.upload()
print(f'✅ Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Cell 3: Mount Google Drive (persistent DB storage) ───────────────────────
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DB_DIR = '/content/drive/MyDrive/FaceID'
os.makedirs(DB_DIR, exist_ok=True)

# Symlink embeddings.json → Google Drive so it persists across sessions
DB_FILE = os.path.join(DB_DIR, 'embeddings.json')
LOCAL_DB = '/content/faceid/embeddings.json'

if not os.path.exists(DB_FILE):
    with open(DB_FILE, 'w') as f:
        f.write('{}')
    print('📁 Created new database at', DB_FILE)
else:
    print(f'📁 Found existing database: {DB_FILE}')
    import json
    with open(DB_FILE) as f:
        existing = json.load(f)
    print(f'   → {len(existing)} registered person(s)')

# Copy DB to local working dir (symlinks can cause issues with Streamlit)
shutil.copy2(DB_FILE, LOCAL_DB)

# Set env var so app.py knows where to persist
os.environ['FACEID_DB_PATH']    = LOCAL_DB
os.environ['FACEID_DB_PERSIST'] = DB_FILE

print('✅ Drive mounted. DB will auto-save to Google Drive.')

In [ ]:
# ── Cell 4: Launch Streamlit + ngrok tunnel ───────────────────────────────────
import os, threading, time
from pyngrok import ngrok, conf

# ── Set your ngrok authtoken (get free one at https://dashboard.ngrok.com) ──
NGROK_TOKEN = ''  # ← PASTE YOUR TOKEN HERE (free at ngrok.com)

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print('⚠️  No ngrok token set. Tunnel may fail or show warning.')
    print('   Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('   Paste it into NGROK_TOKEN above and re-run.')

PORT = 8501

# Kill any existing tunnels
ngrok.kill()

# Launch Streamlit in a background thread
def run_streamlit():
    os.system(f'streamlit run /content/faceid/app.py '
              f'--server.port {PORT} '
              f'--server.headless true '
              f'--server.enableCORS false '
              f'--server.enableXsrfProtection false '
              f'--browser.gatherUsageStats false '
              f'> /content/streamlit.log 2>&1')

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()

# Wait for Streamlit to boot
print('⏳ Starting Streamlit...')
time.sleep(5)

# Open ngrok tunnel
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print()
print('=' * 60)
print(f'🚀  FACE ID SYSTEM IS LIVE!')
print(f'🌐  Open this URL in your browser:')
print(f'    {public_url}')
print('=' * 60)
print()
print('📱  Works on mobile — share the URL with anyone!')
print('💾  Face database auto-saves to Google Drive.')
print('🛑  To stop: Runtime → Interrupt execution')
print()

# DB sync: copy local DB → Drive every 30 seconds
import shutil
DB_PERSIST = os.environ.get('FACEID_DB_PERSIST', '')
LOCAL_DB   = os.environ.get('FACEID_DB_PATH', '/content/faceid/embeddings.json')

def sync_db():
    while True:
        time.sleep(30)
        try:
            if DB_PERSIST and os.path.exists(LOCAL_DB):
                shutil.copy2(LOCAL_DB, DB_PERSIST)
        except Exception as e:
            pass  # Silent — don't break the app

sync_thread = threading.Thread(target=sync_db, daemon=True)
sync_thread.start()

# Keep cell alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    ngrok.kill()
    print('\n🛑 Tunnel closed.')

In [ ]:
# ── Optional Cell 5: View Streamlit logs (for debugging) ─────────────────────
!tail -50 /content/streamlit.log

In [ ]:
# ── Optional Cell 6: Manually back up DB to Drive ────────────────────────────
import shutil, os
LOCAL_DB   = os.environ.get('FACEID_DB_PATH', '/content/faceid/embeddings.json')
DB_PERSIST = os.environ.get('FACEID_DB_PERSIST', '')
if DB_PERSIST:
    shutil.copy2(LOCAL_DB, DB_PERSIST)
    print('✅ Database saved to Google Drive:', DB_PERSIST)
else:
    print('⚠️  No Drive path set. Run Cell 3 first.')